# 02 - Modelo Baseline

Establece el punto de partida minimo que cualquier modelo de ML debe superar.  
Registra los resultados en MLflow para comparacion con los experimentos posteriores.

Baselines evaluados:
1. `DummyClassifier` (estrategia: stratified) - baseline estadistico puro
2. `LogisticRegression` con preprocesamiento correcto (StandardScaler + OneHotEncoder)

**Prerequisito:** Tener los datos descargados en `data/raw/diabetic_data.csv`

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score

from src.data.preprocess import load_raw_data, preprocess
from src.features.engineering import apply_all_features

print('Librerias cargadas correctamente.')

In [ ]:
# Cargar y preprocesar datos
df = load_raw_data()
df = apply_all_features(df)
X_train, X_test, y_train, y_test = preprocess(df)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Tasa positivos train: {y_train.mean():.3f} ({y_train.sum():,} casos)')
print(f'Tasa positivos test:  {y_test.mean():.3f} ({y_test.sum():,} casos)')

In [ ]:
# Separar tipos de columnas para el preprocesador
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Columnas numericas : {len(num_cols)}')
print(f'Columnas categoricas: {len(cat_cols)}')
if cat_cols:
    print(f'  Ejemplos: {cat_cols[:5]}')

In [ ]:
# Configurar MLflow
mlflow.set_tracking_uri('sqlite:///../mlflow.db')
mlflow.set_experiment('diabetes-readmission-prediction')

def log_baseline(name, model, X_train, y_train, X_test, y_test):
    """Entrena un modelo baseline, calcula metricas y las registra en MLflow."""
    with mlflow.start_run(run_name=name):
        mlflow.set_tag('model_type', name)
        mlflow.set_tag('is_baseline', 'true')

        model.fit(X_train, y_train)

        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            auc = 0.5

        y_pred = model.predict(X_test)
        metrics = {
            'roc_auc': auc,
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, 'model')

        print(f'{name}:')
        print(f'  AUC={auc:.4f} | Recall={metrics["recall"]:.4f} | Precision={metrics["precision"]:.4f} | F1={metrics["f1"]:.4f}')
    return metrics

print('Funcion log_baseline lista.')

In [ ]:
# =========================
# Baseline 1: DummyClassifier
# =========================
dummy = DummyClassifier(strategy='stratified', random_state=42)
metrics_dummy = log_baseline(
    'dummy_stratified',
    dummy,
    X_train, y_train,
    X_test, y_test
)

In [ ]:
# =========================
# Baseline 2: Regresion Logistica
# =========================

# Preprocesador: escala numericas y codifica categoricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ],
    remainder='passthrough'
)

# Pipeline completo
lr_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(
        max_iter=500,
        class_weight='balanced',  # critico: compensa el desbalance 1:9
        random_state=42,
        solver='lbfgs',
        n_jobs=-1,
    ))
])

# Entrenar y registrar
metrics_lr = log_baseline(
    'logistic_regression_baseline',
    lr_pipeline,
    X_train, y_train,
    X_test, y_test
)

In [ ]:
# =========================
# Tabla comparativa
# =========================
comparison = pd.DataFrame({
    'Modelo': ['DummyClassifier', 'LogisticRegression', 'XGBoost + Optuna (objetivo)'],
    'ROC-AUC': [round(metrics_dummy['roc_auc'], 4), round(metrics_lr['roc_auc'], 4), '> 0.68'],
    'Recall':  [round(metrics_dummy['recall'], 4),  round(metrics_lr['recall'], 4),  '> 0.65'],
    'F1':      [round(metrics_dummy['f1'], 4),      round(metrics_lr['f1'], 4),      '> 0.35'],
})

print('=== Comparacion de Baselines ===')
print(comparison.to_string(index=False))
print('\nVer todos los runs en http://localhost:5000')

## Conclusiones del Baseline

- El `DummyClassifier` con AUC ~0.50 es el piso absoluto.
- La `LogisticRegression` con `class_weight='balanced'` logra AUC ~0.64 y recall decente,  
  pero precision baja por el desbalance de clases (ratio 1:9).
- El XGBoost con Optuna debe superar ambos en todas las metricas.
- Si XGBoost no supera la Regresion Logistica en AUC, revisar el pipeline de features.